# NASDAQ-100 Microstructure: Holdout Predictions

**Chapter 20 - Out-of-sample evaluation**

Every number in this case study so far was measured on the validation folds, and every
choice was made by looking at them: which model family, which configuration, how many names
to hold at each decision, how to size them, which risk control to overlay, what to charge for
crossing the spread. A result selected that way cannot also be evidence that the selection was
sound - the ranking and the evidence would be the same measurement.

The holdout is the window nothing has been selected on. This notebook fits the selected
configuration on the history available before that window opens and writes its
predictions over it. [`19_holdout_backtest`](19_holdout_backtest.ipynb) turns those
predictions into a return series with the sizing and the overlay the case study settled
on, and [`20_strategy_analysis`](20_strategy_analysis.ipynb) reads both back.

**What this notebook is careful about**

A holdout prediction is not the validation model scored on a later window. Section 2
fits again, over a training interval that ends before the window opens, and the new
training identity is what makes the refit visible rather than asserted: the identity
covers the CV interval, so a run that came back with the validation training hash would
mean no refit happened. The check is in section 3 and it raises.

**Prerequisites:** [`17_costs`](17_costs.ipynb), which is the last stage that selects.

**Scope:** one training run and one prediction set. No backtest, no selection, no
comparison - those are 19 and 20.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# NASDAQ-100 微觀結構：保留測試期預測

**第 20 章 — 樣本外評估**

本案例至今的每個數字都來自驗證資料折，所有選擇也都參考了這些結果：模型家族、設定、每個決策時點持有的股票數、部位規模、疊加的風險控制，以及跨越買賣價差時應計入的成本。以此方式選出的結果，不能再當作選擇合理的證據，因為排名與證據會是同一項量測。

保留測試期是未曾用來作選擇的期間。本 Notebook 使用該期間開始前可得的歷史資料，重新訓練選定設定，並產生該期間的預測。[`19_holdout_backtest`](19_holdout_backtest.ipynb) 將預測依本案例確定的部位規模及風險覆蓋規則轉為報酬序列，[`20_strategy_analysis`](20_strategy_analysis.ipynb) 再讀取兩者。

**本 Notebook 特別注意的事項**

保留測試期預測不是把驗證期模型直接拿到較晚期間評分。第 2 節會在保留測試期開始前結束的訓練區間重新訓練；新的訓練身分可具體證明重新訓練確實發生，而不只是文字宣稱：身分涵蓋交叉驗證區間，因此若某次執行回傳驗證期的訓練雜湊值，就表示沒有重新訓練。第 3 節會檢查，若有此情況即報錯。

**先備條件：**[`17_costs`](17_costs.ipynb)，這是最後一個執行選擇的階段。

**範圍：**一次訓練與一組預測。不含回測、選擇或比較；這些工作由 19 和 20 負責。
<!-- ML4T_ZH_END c0000-p000 -->


In [1]:
"""NASDAQ-100 Microstructure: Holdout Predictions."""

import pandas as pd
import polars as pl

from case_studies.research import open_study
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.research.models import _family_module, reconstruct_locked_model_request
from case_studies.utils.registry import training_hash_from_spec
from case_studies.utils.strategy_analysis import (
    holdout_generations_to_retire,
    registered_holdout_generations,
    resolve_solvent_carrier,
)
from utils.paths import get_case_study_dir

In [2]:
CASE_STUDY_ID = "nasdaq100_microstructure"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""

In [3]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)

## 1. Which configuration the holdout runs

The holdout runs the configuration the case study reports, resolved through the same
`resolve_solvent_carrier` [`17_costs`](17_costs.ipynb) prices. Resolving it again here
rather than passing it along is deliberate: the two notebooks must agree by construction,
and a hash written down in one and read in the other agrees only until the sweep is
rebuilt.

Nothing about the holdout enters this choice. The selected configuration is the cross-stage
validation rank-1, resolved across every declared label rather than per label, and it was fixed
before this notebook ran. Which stage it comes from is printed below rather than asserted here.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 保留測試期使用哪項設定

保留測試期使用本案例呈報的設定，透過 `resolve_solvent_carrier` 解析取得；這也是 [`17_costs`](17_costs.ipynb) 計價時使用的方法。此處重新解析，而非直接沿用，是刻意設計：兩份 Notebook 必須因解析方式而一致；若一份寫下雜湊值、另一份讀取，它們只會在掃描重新建置之前一致。

保留測試期的資料完全不參與此項選擇。選定設定是跨階段、跨所有已宣告標籤解析出的驗證期第一名，而非每個標籤各自排名；它在本 Notebook 執行前已固定。其來源階段會在下方列印，而非在此預先宣稱。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
print(
    f"Selected configuration: {carrier['val_backtest_hash']}  stage={carrier['val_stage']}  "
    f"family={carrier['family']}  config={carrier['config_name']}  "
    f"label={carrier['label']}"
)
print(
    f"  validation Sharpe {carrier['val_sharpe']:.3f}, max drawdown {carrier['max_drawdown']:.3f}"
)
print(f"  fitted by training run {carrier['training_hash']}")

# Whether this family can be refitted at all, asked before anything reads the window.
#
# Every stage below - the holdout CV derivation, the re-keying, the request, the retirement
# check - assumes the selected configuration is a fit that can be repeated on a later fold.
# The ensemble introduced in `14_backtest` Section 4 is not: it is the mean of twelve gbm
# forecasts, so its holdout counterpart is the mean of those twelve models' *holdout*
# forecasts, which is twelve refits and an average rather than the one refit this notebook
# performs. `case_studies/utils/ensemble.py` refuses the re-key for that reason, and that
# refusal would otherwise arrive several steps in, after the window derivation has run.
#
# Asked of the adapter rather than of a family name, so a family that gains the hooks stops
# being refused without anything here changing.
_carrier_module = _family_module(carrier["family"])
_missing_hooks = [
    hook
    for hook in ("rekey_holdout_spec", "reconstruct_locked_request", "validate_locked_run")
    if not callable(getattr(_carrier_module, hook, None))
]
if _missing_hooks:
    msg = (
        f"the selected configuration is {carrier['family']}/{carrier['config_name']}, and that "
        f"family cannot be refitted on the holdout fold: {_carrier_module.__name__} implements "
        f"none of {_missing_hooks}. For the mean-forecast ensemble this is not an oversight in "
        "the adapter - an ensemble has no fit of its own, so its holdout forecast is the mean of "
        "its members' holdout forecasts and producing it means refitting every member and "
        "averaging the results under a new ensemble identity. That is a stage this case study "
        "does not have. Nothing has been written and the window has not been read."
    )
    raise NotImplementedError(msg)

Selected configuration: a2d22880e93f  stage=signal  family=gbm  config=default_multiclass  label=fwd_dir_15m
  validation Sharpe 2.416, max drawdown -0.082
  fitted by training run 7a0c26a08db7


The checkpoint is part of the configuration. Where a family publishes a prediction set per
checkpoint on a declared schedule, the selected configuration's prediction set names one of them,
and refitting without it would produce a model at the end of training rather than the one that
was ranked. A family with no checkpoint dimension stores NULL in both columns and carries that
NULL through unchanged.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
檢查點是設定的一部分。若某模型家族依既定時程為每個檢查點發布一組預測，選定設定的預測集就會指向其中一個；若重新訓練時沒有沿用該檢查點，得到的便是訓練結束時的模型，而非先前參與排名的模型。沒有檢查點維度的模型家族在兩個欄位都儲存 NULL，並原樣沿用這個 NULL。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
validation_prediction = study.results.open(carrier["val_prediction_hash"])
prediction_record = validation_prediction.registry_record()
CHECKPOINT_KIND = prediction_record["checkpoint_kind"]
CHECKPOINT_VALUE = prediction_record["checkpoint_value"]
print(f"Checkpoint: {CHECKPOINT_KIND}={CHECKPOINT_VALUE}")

Checkpoint: iteration=450


## 2. The window, and the model that is allowed to see it

The holdout window is not a choice made here. It is `evaluation.holdout_start` and
`evaluation.holdout_end` from this case study's own `setup.yaml`, read through the same
`canonical_window` the fold derivation and the backtest slice both go through, so the three
cannot disagree.

The training interval is everything available before that window, bounded above by a label
buffer - and **the buffer is not the selected label's.** `build_holdout_cv` takes the widest
buffer any of this case study's labels declares, which here is `61min` from `fwd_ret_60m`, not
the primary `fwd_ret_15m`'s `16min`. The reason is that the holdout fold is one fold: a
fold-scoped temporal artifact carries a single set of boundaries, every label's holdout model
is fitted on features carrying them, so the geometry has to be label-independent and the
widest is the only choice that leaks for no label. A fold built on the sixteen-minute buffer
and handed to the sixty-minute model would give it training rows whose features saw
forty-five minutes past its own `train_end` - the leak the buffer exists to prevent, arriving
through the feature rather than the label.


<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 評估期間，以及模型可看到的資料

保留測試期不是在此選定的。它的 `evaluation.holdout_start` 與 `evaluation.holdout_end` 取自本案例自身的 `setup.yaml`，並透過資料折推導與回測切片共用的 `canonical_window` 讀取，因此三者不會不一致。

訓練區間涵蓋該期間之前所有可得資料，並在終點前留出標籤緩衝區；而且**緩衝區不是只依選定標籤決定。** `build_holdout_cv` 會取本案例所有標籤所宣告的最寬緩衝區，此處是 `61min`，來自 `fwd_ret_60m`，而非主要 `fwd_ret_15m` 的 `16min`。原因是保留測試資料折只有一折：限定於資料折的時間構件只有一組邊界，各標籤的保留測試期模型都依帶有相同邊界的特徵訓練，因此時間結構必須與標籤無關；只有採用最寬緩衝區，才不會讓任何標籤洩漏。若用十六分鐘緩衝區建立資料折，再交給六十分鐘模型，模型就會取得特徵已看到自身 `train_end` 之後四十五分鐘的訓練資料列；這正是緩衝區要防止的洩漏，只是它從特徵而非標籤進入。
<!-- ML4T_ZH_END c0008-p000 -->

**The width is minutes and it is doing the same work a long one does.** Sixty-one minutes
against a window opening on 2021-07-01 looks like nothing beside the ETF study's twenty-one
sessions, and it is the same leak if dropped: a training set running to the first bar of the
window would be fitted on labels that resolve inside it. Each label declares its own
(`fwd_ret_5m: 6min`, `fwd_ret_15m` and `fwd_dir_15m: 16min`, `fwd_ret_60m: 61min`) rather than
inheriting the primary's, and each is a horizon plus one bar because the horizon alone does
not describe the width of the window the outcome resolves over. The derivation refuses to
default any of them.

Everything else about the configuration is carried across unchanged, and the fields that
cannot be - the eligibility manifest, and any parameter this family resolves from a fold's
own training rows - are recomputed against the holdout fold. Carrying those forward would fit
a model keyed to the validation folds and call it a retrain.

<!-- ML4T_ZH_BEGIN c0008-p001 -->
**緩衝區寬度以分鐘計算，作用與較長的緩衝區相同。** 對 2021-07-01 開始的期間而言，六十一分鐘看似遠短於 ETF 研究的二十一個交易時段，但若省略，仍會發生同樣的洩漏：若訓練集一直延伸至保留測試期第一根時間棒，就會用在該期間內才確定結果的標籤訓練。每個標籤都宣告自身的設定（`fwd_ret_5m: 6min`、`fwd_ret_15m`，以及 `fwd_dir_15m: 16min`、`fwd_ret_60m: 61min`），而非繼承主要標籤的設定；各設定都是前瞻期間加上一根時間棒，因為僅有前瞻期間不足以描述結果確定所需的視窗寬度。推導流程拒絕為任何一項使用預設值。

設定的其餘部分原樣沿用；無法直接沿用的欄位——符合條件清單，以及模型家族從資料折自身訓練資料列推導出的參數——則依保留測試資料折重新計算。若直接沿用，就會訓練出以驗證資料折為鍵的模型，卻稱之為重新訓練。
<!-- ML4T_ZH_END c0008-p001 -->


In [6]:
observation_timeline = (
    pl.read_parquet(study.root / "labels" / f"{carrier['label']}.parquet")
    .get_column("timestamp")
    .unique()
    .sort()
    .to_list()
)
validation_spec = study.results.open(carrier["training_hash"]).spec()
holdout_spec = build_holdout_training_spec(
    study,
    validation_spec,
    timeline=observation_timeline,
    case_study=CASE_STUDY_ID,
)

fold = holdout_spec["computation"]["cv"]["folds"][0]
print(f"Holdout fold {fold['fold']}")
print(f"  trains  {fold['train_start']} -> {fold['train_end']}")
print(f"  predicts {fold['val_start']} -> {fold['val_end']}")
print(f"  label buffer: {holdout_spec['computation']['cv']['request']['label_buffer']}")

# The validation folds are what the buffer is measured against, and the last of them ends
# before the holdout opens. Both are printed so a reader can see the gap, and the gap is also
# checked, because a reader is not what runs this.
#
# The check is not a tautology, which is the reason it is here rather than left to the two
# declarations agreeing. `fold["val_start"]` comes from `evaluation.holdout_start` in today's
# `setup.yaml`; `latest_validation_end` comes from the SELECTED CONFIGURATION'S OWN training
# spec, which was registered whenever that configuration was fitted and is not re-derived. The
# carrier pool never retires a row - `published_members_at(member_kind="backtest")` is None for
# this case study, so every backtest ever registered stays selectable - so the resolver can
# hand this notebook a configuration whose folds were built under an earlier window. If that
# window reached past today's `holdout_start`, the configuration was already evaluated on part
# of the period this notebook is about to call out-of-sample, and nothing else would say so:
# the seal below is measured from the holdout's own start and is satisfied either way.
#
# Measured 2026-09-13 on the current registry: latest validation evaluation ends
# 2021-06-30 15:43:00 and the holdout opens 2021-07-01, so this passes today and the refusal
# is for the generation that does not.
#
# Compared as timestamps rather than as strings. The two are rendered by different code -
# `_boundary_iso` writes a midnight boundary as a bare date, and a fold's `val_end` carries a
# time - so `"2021-06-30 15:43:00" < "2021-07-01"` is true by the accident that a space sorts
# below a digit, and would stop being true the moment either renderer changed. Both are put on
# the same naive panel clock first, since the derivation localizes the declared window to the
# panel's own zone and the fold boundaries come off that panel.
validation_folds = validation_spec["computation"]["cv"]["folds"]
latest_validation_end = max(str(entry["val_end"]) for entry in validation_folds)
print(f"Validation folds: {len(validation_folds)}, latest evaluation end {latest_validation_end}")
print(f"Holdout training ends {fold['train_end']}, holdout opens {fold['val_start']}")


def _on_naive_panel_clock(moment: str) -> "pd.Timestamp":
    """The moment as the panel keeps it, with any zone dropped rather than converted."""
    stamp = pd.Timestamp(moment)
    return stamp.tz_localize(None) if stamp.tzinfo is not None else stamp


_validation_closed = _on_naive_panel_clock(latest_validation_end)
_holdout_opened = _on_naive_panel_clock(str(fold["val_start"]))
if _validation_closed >= _holdout_opened:
    msg = (
        f"the selected configuration {carrier['family']}/{carrier['config_name']} on "
        f"{carrier['label']} was evaluated to {latest_validation_end}, and the holdout opens "
        f"{fold['val_start']}. Its validation window reaches into the period this notebook "
        "would report as out-of-sample, so the holdout number it produced would not be one. "
        "That configuration's folds predate the current evaluation.holdout_start rather than "
        "disagreeing with it: a backtest row is never retired, so the resolver can still "
        "select a generation built under an earlier window. Re-fit it under the current "
        "window, or restrict selection so it cannot carry. Nothing has been written."
    )
    raise RuntimeError(msg)

Holdout fold 2
  trains  2020-01-02T09:30:00 -> 2021-06-30T14:42:00
  predicts 2021-07-01 -> 2021-12-31T15:43:00
  label buffer: 61min
Validation folds: 2, latest evaluation end 2021-06-30 15:43:00
Holdout training ends 2021-06-30T14:42:00, holdout opens 2021-07-01


## 3. Fit, and register the predictions

`reconstruct_locked_model_request` builds the request from the spec above. Its name comes
from a locked holdout path this case study does not use; it takes a training specification
and a checkpoint, not a lock, and it is used here because it is the one call that refuses a
request that is not exactly the spec it was handed - the training identity, the checkpoint
schedule, the feature lineage and the runtime parameters are all checked before anything is
fitted.

The training identity below is new. It has to be: it covers the CV interval, and the holdout
fold is not one of the validation folds. A run that came back with the validation training
hash would mean the refit did not happen, so that is checked rather than assumed.

**The window carries one configuration, and this notebook has no way past that.** The check below
is on the selected configuration rather than on the notebook, and it has exactly two outcomes.
With the selected configuration unchanged this is an idempotent replay: the derivation is
deterministic and the training identity covers it, so the same identity comes back and the fit is
served from the registry, which is why re-running the notebook is free and safe. With the
selected configuration changed it refuses, names both configurations, and stops.


<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 3. 訓練並登錄預測

`reconstruct_locked_model_request` 根據上方規格建立請求。其名稱來自本案例未使用的鎖定保留測試路徑；此處它接收的是訓練規格與檢查點，而非鎖定資訊。使用它的原因是：只有這個呼叫會拒絕與收到規格不完全一致的請求；在開始任何訓練前，它會檢查訓練身分、檢查點時程、特徵來源鏈及執行時參數。

下方的訓練身分是新的，而且必須是新的：它涵蓋交叉驗證區間，而保留測試資料折並非任何驗證資料折。若某次執行回傳驗證期的訓練雜湊值，就表示沒有重新訓練；因此此處會實際檢查，而非假設。

**這個期間只容許一項設定，本 Notebook 無法繞過此限制。** 下方檢查針對選定設定，而非 Notebook 本身，且只有兩種結果。若選定設定不變，這是冪等的重播：推導過程是確定性的，訓練身分也涵蓋它，因此會回傳相同身分，並從登錄表取得已訓練的模型；所以重新執行 Notebook 不需重新訓練且安全。若選定設定改變，檢查會拒絕、列出兩項設定，然後停止。
<!-- ML4T_ZH_END c0010-p000 -->

It refuses rather than offering a replacement switch, and the reason is that a replacement
would not be one. Deleting the earlier generation's rows does not undo having observed its
result: the selection that produced the new configuration may have been informed by the old
holdout number, and no deletion reaches that. A switch here would let the case study take a
second look at the window while leaving a registry that shows only one, which is the
specific thing that would make the out-of-sample claim false rather than merely weak.

<!-- ML4T_ZH_BEGIN c0010-p001 -->
它會拒絕請求，而非提供替換開關，因為替換並不能讓評估重新變成第一次。刪除前一版本的資料列，並不能抹去已看過其結果的事實：產生新設定的選擇，可能已受到舊保留測試期數值影響，而刪除動作無法消除此影響。若此處提供開關，案例研究就能再看該期間一次，登錄表卻仍只顯示一次；這會直接使樣本外主張失實，而不只是削弱其可信度。
<!-- ML4T_ZH_END c0010-p001 -->


In [7]:
holdout_training_hash = training_hash_from_spec(holdout_spec)
this_generation = (holdout_training_hash, (CHECKPOINT_KIND, CHECKPOINT_VALUE))
retire = holdout_generations_to_retire(CASE_DIR, this_generation=this_generation)
# A row whose training run records no CV split cannot be shown either way, and deleting on
# that would discard a result nothing has established is wrong. It stops the run instead.
if retire.unattributable:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs record no CV split, "
        "so whether they were refitted for the holdout cannot be established: "
        + ", ".join(
            f"{row['prediction_hash']} (training {row['training_hash']})"
            for row in retire.unattributable
        )
        + ". Establish what produced them before registering another evaluation on the same "
        "window; this notebook will not delete a row it cannot show is not a holdout result."
    )
# A row whose training run declares a non-holdout CV may not be reported as a holdout
# result, and it is also not something to delete unattended: `generate_holdout` refits on a
# holdout fold and then registers the predictions under the VALIDATION training identity, so
# this record covers both a validation-fitted model published over the window and a real
# refit filed under the wrong identity. Nothing owned this before - the filter here was
# `row["refitted"]`, which made exactly these rows invisible to the refusal and to
# everything after it.
if retire.not_out_of_sample:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs declare a CV split "
        "other than the holdout: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in retire.not_out_of_sample
        )
        + ". Each is either a validation-fitted model published over the window, which is "
        "not an out-of-sample result, or a refit registered under its validation training "
        "identity, which the retired `20_strategy_synthesis/holdout.py::generate_holdout` "
        "wrote until it was deleted on 2026-09-12 - and "
        "the registry cannot tell those apart. This notebook has no way past that: "
        "establish which it is and resolve it through the registry's own lifecycle, which "
        "records that the row was retired."
    )
superseded = list(retire.superseded)
if superseded:
    raise RuntimeError(
        "the holdout window already carries a refit of a different configuration: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in superseded
        )
        + f". This run would evaluate {carrier['config_name']} (training "
        f"{holdout_training_hash}, checkpoint {CHECKPOINT_KIND}={CHECKPOINT_VALUE}) on the "
        "same window, which would be a second configuration measured on a period this case "
        "study reports as unseen. This notebook has no way past that: deleting the earlier "
        "generation would not undo having observed it, and the selection bias it introduces "
        "is not removed by removing the rows. Either leave the selection where it was, or "
        "retire the earlier evaluation through the registry's own lifecycle, which records "
        "that a second look was taken."
    )

In [8]:
request = reconstruct_locked_model_request(
    study,
    holdout_spec,
    checkpoint_kind=CHECKPOINT_KIND,
    checkpoint_value=CHECKPOINT_VALUE,
)
model_run = request.run()
holdout_prediction = model_run.predictions[0]

if model_run.training.hash == carrier["training_hash"]:
    raise RuntimeError(
        "the holdout refit produced the validation training identity "
        f"{carrier['training_hash']}, which means it did not refit"
    )
print(f"Holdout training run:  {model_run.training.hash}")
print(f"Holdout prediction set: {holdout_prediction.hash}")

      fold 2: training n_train=11,943,604 n_val=3,949,522 trees=500 num_leaves=? obj=multiclass


      fold 2: done in 1080s


Holdout training run:  f434e268a3e5
Holdout prediction set: 345655bde69d


What the prediction set covers, read back from the registry rather than from the request.
The two agree only if the fit published what it declared, and the counts are what a reader
checks the window against. This is a MINUTE panel evaluated on a per-label decision cadence,
so the row count is decision timestamps times the symbols eligible at each - not sessions
times symbols. The session count is printed separately because it is the number that lines up
with `evaluation.holdout_start` and `holdout_end`, and the two are easy to conflate on an
intraday panel.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
預測集涵蓋的範圍，是從登錄表讀回，而非根據請求推斷。只有訓練流程確實發布宣告的內容，兩者才會一致；資料筆數則供讀者核對評估期間。這是以**分鐘**為單位、依各標籤的決策頻率評估的資料面板，因此資料列數等於決策時間戳記數乘上各時點符合條件的股票數，而不是交易時段數乘上股票數。交易時段數另行列印，因為它是能與 `evaluation.holdout_start` 及 `holdout_end` 對應的數字；在日內資料面板上，兩者很容易混淆。
<!-- ML4T_ZH_END c0013-p000 -->


In [9]:
record = holdout_prediction.registry_record()
predictions = holdout_prediction.load()
print(
    f"split={record['split']}  checkpoint={record['checkpoint_kind']}={record['checkpoint_value']}"
)
print(
    f"rows={predictions.height:,}  "
    f"decision timestamps={predictions['timestamp'].n_unique():,}  "
    f"sessions={predictions['timestamp'].dt.date().n_unique():,}"
)
print(
    f"  {predictions['timestamp'].min()} -> {predictions['timestamp'].max()}, "
    f"{predictions['symbol'].n_unique():,} symbols"
)

split=holdout  checkpoint=iteration=450
rows=3,949,522  decision timestamps=39,884  sessions=128


  2021-07-01 10:31:00+00:00 -> 2021-12-31 15:43:00+00:00, 102 symbols


Every holdout prediction set the registry holds, and whether the model behind it was
refitted for the window. All of them are listed rather than one silently preferred, because
the registry is immutable and a reader looking at it later will see whatever is there. A row
marked VALIDATION-FITTED is not an out-of-sample result whatever its numbers say.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
列出登錄表中所有保留測試期預測集，以及其模型是否為該期間重新訓練。此處全部列出，而非悄悄偏好其中一項，因為登錄表不可變；之後查看的人會看到實際留存的所有內容。標示為 VALIDATION-FITTED 的資料列，不論數字如何，都不是樣本外結果。
<!-- ML4T_ZH_END c0015-p000 -->


In [10]:
for row in registered_holdout_generations(CASE_DIR):
    note = (
        "refitted for the holdout" if row["refitted"] else "VALIDATION-FITTED - not out of sample"
    )
    print(
        f"  {row['prediction_hash']}  training={row['training_hash']}  {row['config_name']}  {note}"
    )

  345655bde69d  training=f434e268a3e5  default_multiclass  refitted for the holdout


## What this notebook establishes, and what it does not

It establishes one thing: a prediction set over the holdout window, produced by the
configuration this case study selected, fitted on data that ends a full label horizon before
the window opens. That is a precondition for an out-of-sample claim, not the claim itself.
Nothing here says whether the predictions are any good - they have not been scored, sized or
traded.

It does not make the holdout a fresh test in the strict sense. The configuration reached this
notebook through a selection made on the validation folds, and this window is being used once
per configuration that gets here. What it does remove is the specific circularity of scoring
a validation-fitted model on the period meant to judge it.

Re-running this notebook is free: the same configuration re-derives the same training identity and
the fit is served from the registry. Evaluating a DIFFERENT configuration is not, and is
refused here. If a later pass finds the selection was wrong, that is a question for the
registry's lifecycle, which records that a second look was taken - not something to settle by
deleting rows until the registry agrees.

**Next:** [`19_holdout_backtest`](19_holdout_backtest.ipynb).

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 本 Notebook 確立了什麼，又未確立什麼

它只確立一件事：本案例選定設定在保留測試期間產生了一組預測，其模型使用的資料在該期間開始前至少一個完整標籤前瞻期間就已結束。這是提出樣本外主張的先決條件，而非主張本身。此處尚未說明預測是否良好，因為還未評分、配置部位規模或交易。

嚴格來說，它也無法讓保留測試期變成全新的測試。設定是經由驗證資料折的選擇來到此處，而每個進入此流程的設定都會使用此期間一次。它排除的是一種特定循環：以驗證期訓練的模型，直接在原本用來檢驗它的期間評分。

重新執行本 Notebook 不增加成本：相同設定會重新推導出相同訓練身分，模型則由登錄表提供。但評估**不同**設定不是無成本的，因此此處會拒絕。若後續發現選擇有誤，那是登錄表生命週期應處理的問題，並須記錄曾進行第二次檢視；不能靠反覆刪除資料列，讓登錄表看起來符合預期。

**下一步：**[`19_holdout_backtest`](19_holdout_backtest.ipynb)。
<!-- ML4T_ZH_END c0017-p000 -->
